# UEBA Behavioral Analysis

Profile data-access behavior and flag users with unusual after-hours or weekend activity.

In [ ]:
import os
from datetime import datetime, timedelta
import clickhouse_connect
import pandas as pd
import plotly.express as px
from scipy import stats
from clario360 import Client

DAYS_BACK = int(os.environ.get('CLARIO360_UEBA_WINDOW_DAYS', '30'))
sdk = Client.from_env()
clickhouse = clickhouse_connect.get_client(
    host=os.environ['CLICKHOUSE_HOST'],
    port=int(os.environ.get('CLICKHOUSE_PORT', '9000')),
    database=os.environ.get('CLICKHOUSE_DATABASE', 'clario360'),
    username=open('/etc/clario360/credentials/clickhouse_user').read().strip(),
    password=open('/etc/clario360/credentials/clickhouse_password').read().strip(),
)

## Query access telemetry

The query is read-only and the notebook records the access for auditability.

In [ ]:
query = '''
SELECT user_email, action, resource_type, resource_id, toHour(timestamp) AS hour_of_day,
       toDayOfWeek(timestamp) AS day_of_week, toDate(timestamp) AS event_date, timestamp
FROM security_events
WHERE tenant_id = %(tenant_id)s
  AND timestamp >= now() - INTERVAL %(days)s DAY
  AND action IN ('read', 'query', 'download', 'export')
ORDER BY timestamp DESC
'''
sdk.record_data_query('clickhouse', 'UEBA behavioral baseline query', {'days_back': DAYS_BACK})
df = clickhouse.query_df(query, parameters={'tenant_id': sdk.tenant_id, 'days': DAYS_BACK})
if df.empty:
    raise RuntimeError('No access events available for UEBA analysis.')
df.head(10)

## Build profiles and flag anomalies

This notebook uses simple z-score features so the workflow stays transparent and explainable.

In [ ]:
profiles = df.groupby('user_email').agg(
    total_events=('timestamp', 'count'),
    unique_resources=('resource_id', 'nunique'),
    weekend_events=('day_of_week', lambda s: s.isin([6, 7]).sum()),
    after_hours=('hour_of_day', lambda s: ((s < 7) | (s > 19)).sum()),
).reset_index()
profiles['weekend_ratio'] = profiles['weekend_events'] / profiles['total_events']
profiles['after_hours_ratio'] = profiles['after_hours'] / profiles['total_events']
for column in ['unique_resources', 'weekend_ratio', 'after_hours_ratio']:
    profiles[f'{column}_z'] = stats.zscore(profiles[column].fillna(0), nan_policy='omit')
profiles['max_zscore'] = profiles[[c for c in profiles.columns if c.endswith('_z')]].abs().max(axis=1)
profiles['is_anomalous'] = profiles['max_zscore'] >= 3
fig = px.scatter(profiles, x='unique_resources', y='after_hours_ratio', color='is_anomalous', hover_name='user_email', title='UEBA Outlier Review')
fig.show()
profiles[profiles['is_anomalous']].sort_values('max_zscore', ascending=False)